# Exercise 10 – Sparse VARX: Adding Environmental Drivers
### Workshop Tutorial: From Lotka–Volterra to Real Ecological Data

In [ ]:
knitr::opts_chunk$set(
  echo    = TRUE,
  message = FALSE,
  warning = FALSE
)

---

# Background: What is a VARX model?

In Exercise 09 we worked with **sparse VAR** models — models where the only predictors are the system's own past values (*endogenous* variables). But in ecology, populations do not evolve in isolation. Temperature, salinity, oxygen, and nutrients all shape species dynamics from the *outside*.

A **VARX** model extends VAR by including additional *exogenous* variables:

$$Y_t = \underbrace{\Phi_1 Y_{t-1} + \dots + \Phi_p Y_{t-p}}_{\text{endogenous (VAR) part}} + \underbrace{B_1 X_{t-1} + \dots + B_s X_{t-s}}_{\text{exogenous (X) part}} + \varepsilon_t$$

where:

| Symbol | Meaning |
|---|---|
| $Y_t$ | Endogenous variables at time $t$ (e.g. taxon abundances) |
| $X_t$ | Exogenous variables at time $t$ (e.g. temperature, oxygen) |
| $\Phi_l$ | Autoregressive coefficient matrices for lag $l$ |
| $B_l$ | Exogenous effect matrices for lag $l$ |
| $p, s$ | Maximum lag orders for endogenous and exogenous variables |

The key advantage: environmental forcing can be captured *directly* rather than absorbed as noise, leading to better forecasts and more interpretable ecological networks.

---

# Exercise 1 — Recovering Lotka–Volterra Parameters with VARX

## Learning Goals

You will connect a **continuous-time ecological model** (generalized Lotka–Volterra, gLV) to a **discrete-time VARX representation** and estimate its parameters from simulated data. This is a key conceptual bridge: real ecological interaction networks are often described as gLV systems, and sparse VARX gives us a way to *infer* those networks from time series.

---

## The Generalized Lotka–Volterra System

The gLV model describes the dynamics of $d$ interacting species:

$$\frac{dx_i}{dt} = r_i x_i + \sum_{j=1}^{d} \alpha_{ij}\, x_i\, x_j$$

Parameter meanings:

| Parameter | Meaning |
|---|---|
| $r_i > 0$ | Species $i$ grows intrinsically (e.g. prey) |
| $r_i < 0$ | Species $i$ declines intrinsically (e.g. predator without prey) |
| $\alpha_{ij} > 0$ | Species $j$ *promotes* growth of species $i$ |
| $\alpha_{ij} < 0$ | Species $j$ *inhibits* species $i$ |

**From gLV to VARX:** Taking the log-difference of the gLV system gives an approximate discrete-time representation — a VARX(1,1) where:

- **Endogenous** = $\Delta \log x$ (log-differences of abundances)
- **Exogenous** = $x$ itself (raw abundances at the previous step)
- The exogenous coefficient matrix $B$ encodes $\alpha$, scaled by $\Delta t$
- The intercept $\phi_0$ encodes the growth rate $r$, scaled by $\Delta t$

---

## Step 1: Define the true system parameters

We simulate a 3-species predator–prey system. The true parameters define the reference against which we evaluate our estimates.

In [ ]:
parms <- c(
  alpha  =  0.5,   # intrinsic growth rate of prey
  beta1  = -0.5,   # effect of predator 1 on prey
  beta2  = -0.1,   # effect of predator 2 on prey
  delta1 =  0.25,  # benefit of prey to predator 1
  delta2 =  0.2,   # benefit of prey to predator 2
  gamma1 = -1.0,   # intrinsic mortality of predator 1
  gamma2 = -0.8    # intrinsic mortality of predator 2
)

---

## Step 2: Organise the true parameters into r and α

**Task:** Fill in the parameter names to assign the correct values to `r_true`.

In [ ]:
# Growth rate vector: prey grows (positive), predators decay (negative)
r_true <- c(
  Prey  = parms["___"],    # which parameter is the prey's growth rate?
  Pred1 = parms["___"],    # intrinsic rate of predator 1
  Pred2 = parms["___"]     # intrinsic rate of predator 2
)

# Interaction matrix (3x3), starts as all zeros
alpha_true <- matrix(0, nrow = 3, ncol = 3,
                     dimnames = list(c("Prey", "Pred1", "Pred2"),
                                     c("Prey", "Pred1", "Pred2")))

alpha_true["Prey",  "Pred1"] <- parms["beta1"]
alpha_true["Prey",  "Pred2"] <- parms["beta2"]
alpha_true["Pred1", "Prey"]  <- parms["delta1"]
alpha_true["Pred2", "Prey"]  <- parms["delta2"]

cat("True growth rates (r):\n");  print(r_true)
cat("\nTrue interaction matrix (alpha):\n"); print(alpha_true)

> **What to notice:** Most entries in `alpha_true` are zero — the true system is *sparse*. Only prey-predator pairs interact directly.

---

## Step 3: Load the simulated time series

In [ ]:
library(bigtime)

Y <- read.csv("lotka_volterra_sim_2.csv", check.names = FALSE)

cat("Time points:", nrow(Y), "\n")
cat("Species:    ", ncol(Y), "\n")

---

## Step 4: Construct the endogenous and exogenous matrices

This is the key data preparation step connecting the gLV to VARX.

**Task:** Fill in the two transformation steps below.

In [ ]:
dt <- 0.2     # time step of the simulation

n     <- nrow(Y)
Y_log <- ___(Y)                       # log-transform: log(x)

Y_endog <- ___(as.matrix(Y_log))      # first differences of log: gives Δlog(x)
X_exog  <- as.matrix(Y[1:(n - 1), ]) # raw abundances at t-1 (exogenous predictor)

colnames(Y_endog) <- c("logPrey", "logPred1", "logPred2")
colnames(X_exog)  <- c("Prey",    "Pred1",    "Pred2")

cat("Y_endog (log-differences):", nrow(Y_endog), "×", ncol(Y_endog), "\n")
cat("X_exog  (raw abundances): ", nrow(X_exog),  "×", ncol(X_exog),  "\n")

---

## Step 5: Fit the sparse VARX model

### Key function: `sparseVARX()`

`sparseVARX()` from the `bigtime` package estimates a regularized VARX model with separate penalties for $\Phi$ (endogenous) and $B$ (exogenous).

| Argument | Description |
|---|---|
| `Y` | Endogenous time series matrix (T × k) |
| `X` | Exogenous time series matrix (T × q) — same number of rows as `Y` |
| `p` | Maximum lag order for the endogenous part |
| `s` | Maximum lag order for the exogenous part |
| `VARXpen` | Penalty type: `"L1"` or `"HLag"` |
| `VARXlPhiseq` | Grid of λ values to search for $\Phi$ |
| `VARXlBseq` | Grid of λ values to search for $B$ |
| `selection` | `"cv"` for cross-validation |

The fitted object contains:

| Field | Description |
|---|---|
| `$Phihat` | Estimated endogenous coefficient matrix |
| `$Bhat` | Estimated exogenous coefficient matrix |
| `$phi0hat` | Intercept vector (encodes growth rates $r$) |
| `$lambdaPhi_opt` | Optimal λ for the endogenous part |
| `$lambdaB_opt` | Optimal λ for the exogenous part |

**Task:** Fill in `Y`, `X`, `p`, `s`, and `VARXpen`.

In [ ]:
fit_varx <- sparseVARX(
  Y           = ___,                              # endogenous: log-differences
  X           = ___,                              # exogenous: raw abundances
  p           = ___,                              # one lag for endogenous
  s           = ___,                              # one lag for exogenous
  VARXpen     = "___",                            # LASSO penalty
  VARXlPhiseq = seq(10, 100, length.out = 10),
  VARXlBseq   = seq(10, 100, length.out = 10),
  selection   = "cv"
)

cat("Optimal λ for Phi:", fit_varx$lambdaPhi_opt, "\n")
cat("Optimal λ for B:  ", fit_varx$lambdaB_opt,  "\n")

---

## Step 6: Inspect diagnostics and lag structure

In [ ]:
p_cv <- diagnostics_plot(fit_varx, variable = 3)
plot(p_cv)

In [ ]:
lagmatrix(fit_varx, returnplot = TRUE)

> **Reading the VARX lag matrix:** The matrix now has two parts — one for the endogenous (lagged $\Delta\log x$) effects and one for the exogenous ($x$) effects. Non-zero entries in the $B$ part correspond directly to species interaction terms in the gLV model.

---

## Step 7: Recover the gLV parameters

We back-transform the fitted VARX coefficients to the original gLV scale using:

$$\hat{r}_i = \frac{\hat{\phi}_{0,i}}{\Delta t}, \qquad \hat{\alpha} = \frac{\hat{B}}{\Delta t}$$

**Task:** Fill in the two division operations.

In [ ]:
# Growth rates: intercept divided by dt
r_hat <- fit_varx$phi0hat ___ dt

r_compare <- data.frame(
  Species = colnames(Y_endog),
  True_r  = as.numeric(r_true),
  Est_r   = as.numeric(r_hat)
)
print(r_compare)

In [ ]:
# Interaction matrix: sum B matrices across lags, then divide by dt
k <- ncol(X_exog)
q <- fit_varx$s

B_total <- matrix(0, nrow = k, ncol = k)
for (lag in 1:q) {
  B_lag   <- fit_varx$Bhat[, ((lag - 1) * k + 1):(lag * k)]
  B_total <- B_total + B_lag
}

# Divide by dt to recover alpha
alpha_hat <- B_total ___ dt

alpha_compare <- data.frame(
  Interaction = outer(rownames(alpha_true), colnames(alpha_true),
                      paste, sep = " <- ") |> as.vector(),
  True_alpha  = as.vector(alpha_true),
  Est_alpha   = as.vector(alpha_hat)
)
print(alpha_compare)

> **What to check:**
> - Are the *signs* of estimated $r$ correct (positive for prey, negative for predators)?
> - Does the model correctly shrink zero entries in `alpha_true` toward zero?
> - Why might small true values like $\beta_2 = -0.1$ be harder to recover precisely?

---

# Exercise 2 — Sparse VAR vs. VARX on Baltic Sea Data

## Learning Goals

You will apply both a sparse VAR and two sparse VARX models (LASSO and HLag) to a real marine time series and evaluate whether adding environmental variables improves the model.

---

## Step 1: Load and inspect the data

In [ ]:
time_series <- as.matrix(read.csv("time_series_balticsea.csv"))
metadata    <- as.matrix(read.csv("metadata_time_series_imp.csv"))

colnames(metadata) <- c("Temperature", "oxy_ml_per_L", "Chl",
                        "pressure", "NO3_LOD2", "NO2_LOD2",
                        "Silicate_LOD2", "PO4_LOD2")

cat("Biological time series:", nrow(time_series), "×", ncol(time_series), "\n")
cat("Environmental metadata:", nrow(metadata),    "×", ncol(metadata),    "\n")

> **Reminder:** Scale both matrices before fitting. Without scaling, variables with larger absolute values would dominate the regularization penalty.

---

## Step 2: Fit the LASSO-penalized sparse VARX

**Task:** Fill in `Y`, `X` (Temperature and oxygen from metadata), `VARXpen`, and `selection`.

In [ ]:
fit_varx <- sparseVARX(
  Y         = scale(___),                                           # biological time series
  X         = scale(metadata[, c("___", "___")]),   # Temperature and oxygen
  p         = 3,
  VARXpen   = "___",
  VARXlBseq = seq(0.1, 10, length.out = 10),
  selection = "___"
)

cat("Optimal λ for Phi:", fit_varx$lambdaPhi_opt, "\n")

---

## Step 3: Extract the optimal CV-MSFE for the LASSO VARX

The model stores a 2D grid of MSFE values — one for every ($\lambda_\Phi$, $\lambda_B$) combination. We find the grid point that matches the one-SE-optimal penalties.

In [ ]:
idx_lasso  <- which(fit_varx$lambdaB   == fit_varx$lambdaB_SEopt &
                    fit_varx$lambdaPhi == fit_varx$lambdaPhi_SEopt)
best_lasso <- idx_lasso[which.min(fit_varx$MSFEcv[idx_lasso])]
cv_lasso   <- fit_varx$MSFEcv[best_lasso]

cat("CV-MSFE (LASSO VARX):", round(cv_lasso, 4), "\n")

> **What is the 1-SE rule?** Instead of the absolute minimum CV error, this rule picks the *simplest* model whose CV error is within one standard error of the minimum. This prevents chasing marginal improvements.

---

## Step 4: Diagnostics and lag structure — LASSO VARX

In [ ]:
diagnostics_plot(fit_varx)

In [ ]:
l_varx <- lagmatrix(fit_varx, returnplot = TRUE)

---

## Step 5: Fit the HLag-penalized sparse VARX

**Task:** Same setup as Step 2, but change `VARXpen` to `"HLag"`.

In [ ]:
fit_varx_h <- sparseVARX(
  Y         = scale(time_series),
  X         = scale(metadata[, c("Temperature", "oxy_ml_per_L")]),
  p         = 3,
  VARXpen   = "___",          # hierarchical lag penalty
  VARXlBseq = seq(0.1, 10, length.out = 10),
  selection = "cv"
)

---

## Step 6: Extract CV-MSFE for HLag VARX

In [ ]:
idx_hlag  <- which(fit_varx_h$lambdaB   == fit_varx_h$lambdaB_SEopt &
                   fit_varx_h$lambdaPhi == fit_varx_h$lambdaPhi_SEopt)
best_hlag <- idx_hlag[which.min(fit_varx_h$MSFEcv[idx_hlag])]
cv_hlag   <- fit_varx_h$MSFEcv[best_hlag]

cat("CV-MSFE (HLag VARX):", round(cv_hlag, 4), "\n")

---

## Step 7: Diagnostics and lag structure — HLag VARX

In [ ]:
diagnostics_plot(fit_varx_h)

In [ ]:
l_varx_h <- lagmatrix(fit_varx_h, returnplot = TRUE)

---

## Step 8: Fit the sparse VAR baseline (no exogenous variables)

**Task:** Fit a plain sparse VAR with `p = 3` and cross-validation (no `X` argument).

In [ ]:
fit_var <- sparseVAR(
  Y         = scale(___),   # biological time series only
  p         = ___,
  selection = "cv"
)

plot_cv(fit_var)

---

## Step 9: Extract CV-MSFE for the VAR baseline

In [ ]:
idx_var  <- which(fit_var$lambdas == fit_var$lambda_opt)
best_var <- idx_var[which.min(fit_var$MSFEcv[idx_var])]
cv_var   <- fit_var$MSFEcv[best_var]

cat("CV-MSFE (sparse VAR, no exogenous):", round(cv_var, 4), "\n")

---

## Step 10: Diagnostics and lag structure — VAR baseline

In [ ]:
diagnostics_plot(___)

In [ ]:
l_var <- lagmatrix(___, returnplot = TRUE)

---

## Step 11: Compare model complexity

**Task:** Count non-zero coefficients in Phihat for each model and in Bhat for the VARX models.

In [ ]:
nz_var      <- sum(fit_var$Phihat  ___ 0)    # != 0 counts non-zero entries
nz_varx_phi <- sum(fit_varx$Phihat ___ 0)
nz_varx_B   <- sum(fit_varx$Bhat   ___ 0)

cat("Non-zero coefficients:\n")
cat(sprintf("  sparse VAR — Phi:           %d\n", nz_var))
cat(sprintf("  LASSO VARX — Phi:           %d\n", nz_varx_phi))
cat(sprintf("  LASSO VARX — B (exogenous): %d\n", nz_varx_B))

---

## Step 12: Compare lag matrices — VAR vs. VARX

In [ ]:
heatmap(
  as.matrix(l_var$LPhi) - as.matrix(l_varx$LPhi),
  main  = "Δ Active lags: VAR minus LASSO VARX",
  scale = "none",
  col   = colorRampPalette(c("#b2182b", "white", "#2166ac"))(50)
)

> **Blue cells** = lags active in VAR but dropped in VARX. These endogenous links disappear once environmental covariates absorb the variance they were explaining.

---

## Step 13: Coefficient heatmaps

In [ ]:
library(ggplot2); library(tidyr); library(dplyr); library(tibble); library(patchwork)

heatmap_coef <- function(mat, title, show_y = TRUE, fixed = TRUE) {
  p <- as.data.frame(mat) |>
    rownames_to_column("response") |>
    pivot_longer(-response, names_to = "predictor", values_to = "coef") |>
    ggplot(aes(x = predictor, y = response, fill = abs(coef))) +
    geom_tile(color = "grey80", linewidth = 0.2) +
    geom_text(aes(label = ifelse(abs(coef) > 0.01, round(abs(coef), 1), "")),
              size = 2, color = "grey20") +
    scale_fill_gradient(low = "white", high = "#08519c", name = NULL,
                        guide = guide_colorbar(barwidth = 6, barheight = 0.4,
                                               ticks = FALSE)) +
    scale_x_discrete(position = "top") +
    scale_y_discrete(limits = rev) +
    theme_minimal(base_size = 8) +
    theme(axis.text.x    = element_text(angle = 90, vjust = 0.5, hjust = 0, size = 6),
          axis.text.y    = element_text(size = 6),
          axis.title     = element_blank(),
          panel.border   = element_rect(color = "black", fill = NA, linewidth = 0.8),
          panel.grid     = element_blank(),
          legend.position = "top",
          plot.title     = element_text(size = 8, hjust = 0.5)) +
    labs(title = title)
  if (fixed)   p <- p + coord_fixed()
  if (!show_y) p <- p + theme(axis.text.y = element_blank())
  p
}

**Task:** Extract the Temperature column (column 1) and Oxygen column (column 2) from `l_varx$LB` and pass them to `heatmap_coef()`.

In [ ]:
p1 <- heatmap_coef(l_varx$LPhi, "(a) LASSO: Taxa interactions")

# Extract Temperature column from the exogenous lag matrix
mat_temp <- l_varx$LB[, ___, drop = FALSE]
colnames(mat_temp) <- "Temperature"
p2 <- heatmap_coef(mat_temp, "(b) LASSO: Temperature effect")

# Extract Oxygen column
mat_oxy <- l_varx$LB[, ___, drop = FALSE]
colnames(mat_oxy) <- "Oxygen"
p3 <- heatmap_coef(mat_oxy, "(c) LASSO: Oxygen effect")

p1; p2; p3

---

## Step 14: In-sample MSE comparison

In [ ]:
mse_var  <- mean(residuals(fit_var)^2)
mse_varx <- mean(residuals(fit_varx)^2)

cat("In-sample MSE:\n")
cat(sprintf("  sparse VAR : %.4f\n", mse_var))
cat(sprintf("  LASSO VARX : %.4f\n", mse_varx))

---

## Step 15: Residual ACF comparison

In [ ]:
par(mfrow = c(1, 2))
acf(residuals(fit_var)[,  1], main = "ACF: sparse VAR (taxon 1)",   lag.max = 20)
acf(residuals(fit_varx)[, 1], main = "ACF: LASSO VARX (taxon 1)",  lag.max = 20)
par(mfrow = c(1, 1))

---

## Step 16: Ljung–Box test for white-noise residuals

The Ljung–Box test checks formally whether residuals have remaining autocorrelation. Null hypothesis: residuals are white noise. A **p-value > 0.05** = no significant autocorrelation.

### Key function: `Box.test()`

| Argument | Description |
|---|---|
| `x` | A univariate time series vector |
| `lag` | Number of lags included in the test (rule of thumb: ~$\log(T)$) |
| `type` | Use `"Ljung-Box"` for diagnostic checking |

**Task:** Fill in the `lag` and `type` arguments.

In [ ]:
check_residuals <- function(res_matrix, model_name) {
  p_values <- apply(res_matrix, 2, function(x) {
    Box.test(x, lag = ___, type = "___")$p.value   # fill in lag and type
  })
  data.frame(
    Model          = model_name,
    Variable       = colnames(res_matrix),
    P_Value        = round(p_values, 4),
    Is_White_Noise = p_values > 0.05
  )
}

test_var  <- check_residuals(residuals(fit_var),  "VAR")
test_varx <- check_residuals(residuals(fit_varx), "VARX")

comparison_table <- rbind(test_var, test_varx)
print(comparison_table)

In [ ]:
cat("Variables with white-noise residuals:\n")
cat(sprintf("  sparse VAR : %d / %d\n",
            sum(comparison_table$Is_White_Noise[comparison_table$Model == "VAR"]),
            ncol(time_series)))
cat(sprintf("  LASSO VARX : %d / %d\n",
            sum(comparison_table$Is_White_Noise[comparison_table$Model == "VARX"]),
            ncol(time_series)))

> **Discussion questions:**
> 
> 1. Does VARX have more white-noise residuals than VAR? What does this tell us about the role of environmental drivers?
> 2. Which model would you choose for forecasting? For network reconstruction?

---

# Exercise 3 — Exploring Additional Covariates

## Learning Goals

You extend the VARX model by including Chlorophyll as a third environmental covariate and evaluate whether it adds meaningful predictive information.

---

## Step 1: Fit a VARX with three covariates

**Task:** Add `"Chl"` as a third covariate alongside Temperature and Oxygen.

In [ ]:
fit_varx_chl <- sparseVARX(
  Y         = scale(time_series),
  X         = scale(metadata[, c("Temperature", "oxy_ml_per_L", "___")]),  # add Chl
  p         = 3,
  selection = "cv",
  VARXlBseq = seq(0.1, 10, length.out = 10)
)

---

## Step 2: CV plot and optimal MSFE

In [ ]:
plot_cv(fit_varx_chl)

In [ ]:
idx_chl  <- which(fit_varx_chl$lambdaB   == fit_varx_chl$lambdaB_SEopt &
                  fit_varx_chl$lambdaPhi == fit_varx_chl$lambdaPhi_SEopt)
best_chl <- idx_chl[which.min(fit_varx_chl$MSFEcv[idx_chl])]
cv_chl   <- fit_varx_chl$MSFEcv[best_chl]

cat("CV-MSFE (Temp + Oxy + Chl):", round(cv_chl, 4), "\n")
cat("CV-MSFE (Temp + Oxy only): ", round(cv_lasso, 4), "\n")

---

## Step 3: Diagnostics and lag structure

In [ ]:
diagnostics_plot(___)

In [ ]:
l_varx_chl <- lagmatrix(___, returnplot = TRUE)

---

## Step 4: Summary comparison across all models

**Task:** Complete the `Nonzero_B` entry for `fit_varx_chl` and fill in the missing CV-MSFE values.

In [ ]:
nz_varx_h_phi <- sum(fit_varx_h$Phihat != 0)
nz_varx_h_B   <- sum(fit_varx_h$Bhat   != 0)
nz_chl_phi    <- sum(fit_varx_chl$Phihat != 0)
nz_chl_B      <- sum(fit_varx_chl$Bhat   != 0)

summary_table <- data.frame(
  Model       = c("sparse VAR", "VARX LASSO (T+O)", "VARX HLag (T+O)", "VARX LASSO (T+O+Chl)"),
  CV_MSFE     = round(c(___, ___, ___, ___), 4),   # fill in from earlier steps
  Nonzero_Phi = c(nz_var, nz_varx_phi, nz_varx_h_phi, nz_chl_phi),
  Nonzero_B   = c(NA,     nz_varx_B,   nz_varx_h_B,   nz_chl_B)
)

print(summary_table)

---

## Final Discussion

1. **Does adding exogenous variables improve prediction?** Compare CV-MSFE across models. Does Chlorophyll still help beyond Temperature and Oxygen?

2. **Effect on sparsity:** Does VARX have fewer non-zero Phi entries than VAR? Why might exogenous variables *reduce* the number of endogenous links?

3. **Lagged environmental effects:** From the lag matrices, are Temperature and Oxygen effects immediate (Lag 1) or delayed? What biological mechanism could explain a lagged response?

4. **Overfitting risk:** At what point does adding more covariates stop improving CV-MSFE? How would you select covariates in a real-world application?

5. **VAR vs. VARX for network reconstruction:** Is a link in the VAR Phi matrix necessarily a direct biological interaction, or could it be a shared environmental response?

---

*End of Exercise 10*